# 4.2 — Clasificación multiclase con features clínicas enriquecidas

**Objetivo**: Evaluar si la incorporación de variables clínicas y etiquetas de cluster
como **features en X** (no en y) mejora la clasificación multiclase.

**Variable objetivo (y)**: 19 clases originales (18 tipos de cáncer + nonMalignant).

**Features en X**:
- ~5440 genes (ENSG*)
- `mal_cluster` (dummy one-hot, 2 columnas)
- `nm_cluster` (dummy one-hot, 2 columnas)
- `Age` (estandarizada, imputada con mediana de train)
- `Sex` (one-hot F/M, imputada con moda de train)

**Prevención de data leakage**: `ClinicalFeaturePreprocessor` dentro del pipeline
(fit en train, transform en test/CV folds).

**Comparación**: vs 4.0 (solo genes) y 4.1 (clusters en y).

In [2]:
from __future__ import annotations
from pathlib import Path
from dataclasses import replace
import logging
import warnings
from sklearn.exceptions import ConvergenceWarning

import pandas as pd
import numpy as np

from time import perf_counter
from tqdm.auto import tqdm

from genomics_dl.models.train_multiclass import (
    MulticlassTrainConfig,
    run_training,
)

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("mlflow").setLevel(logging.WARNING)

### Rutas y carga de datos

In [3]:
DATA_PROCESSED = Path("../data/processed")

# Usamos los parquets clustered (con mal_cluster y nm_cluster)
TRAIN_PATH = DATA_PROCESSED / "gse183635_tep_tpm_train_clustered.parquet"
TEST_PATH  = DATA_PROCESSED / "gse183635_tep_tpm_test_clustered.parquet"

df_train = pd.read_parquet(TRAIN_PATH)
df_test  = pd.read_parquet(TEST_PATH)

print(f"Train: {df_train.shape}")
print(f"Test:  {df_test.shape}")

Train: (1880, 5454)
Test:  (471, 5454)


### Separación genes vs metadatos

In [4]:
metadata_cols = [
    "Sample ID", "Patient_group", "Stage", "Sex", "Age",
    "Sample-supplying institution", "Training series",
    "Evaluation series", "Validation series", "lib.size",
    "classificationScoreCancer", "Class_group",
    "mal_cluster", "nm_cluster",
]

gene_cols = [c for c in df_train.columns if c not in metadata_cols]
print(f"Genes: {len(gene_cols)}")
print(f"Columnas clínicas/cluster: mal_cluster, nm_cluster, Age, Sex")
print(f"\nVerificación:")
print(f"  mal_cluster en train: {df_train['mal_cluster'].notna().sum()} no-NaN")
print(f"  nm_cluster en train:  {df_train['nm_cluster'].notna().sum()} no-NaN")
print(f"  Age NaN en train:     {df_train['Age'].isna().sum()}")
print(f"  Sex n.a. en train:    {(df_train['Sex'] == 'n.a.').sum()}")

Genes: 5440
Columnas clínicas/cluster: mal_cluster, nm_cluster, Age, Sex

Verificación:
  mal_cluster en train: 1302 no-NaN
  nm_cluster en train:  578 no-NaN
  Age NaN en train:     18
  Sex n.a. en train:    14


## Experimentos con features clínicas (sweep)

Ranking:
1) Minimizar `test_cancer_fn`
2) Maximizar `test_cancer_recall_sensitivity`
3) Maximizar `test_f1_macro`

In [5]:
def slugify_token(value):
    return str(value).replace(".", "p").replace("-", "m")

def build_model_name(clf_name, feat_cfg, malignant_weight, variant_tag):
    return "_".join([
        clf_name,
        f"pca{int(feat_cfg['use_pca'])}",
        f"log{int(feat_cfg['selector_on_log'])}",
        f"vq{int(feat_cfg['var_quantile']*100)}",
        f"mw{slugify_token(malignant_weight)}",
        variant_tag,
    ])

def fmt_secs(s: float) -> str:
    s = int(max(0, s))
    h = s // 3600
    m = (s % 3600) // 60
    ss = s % 60
    if h > 0:
        return f"{h:d}h {m:02d}m {ss:02d}s"
    if m > 0:
        return f"{m:d}m {ss:02d}s"
    return f"{ss:d}s"

In [ ]:
# Base config — cluster_as_feature=True para mover clusters a X
base_cfg = MulticlassTrainConfig(
    train_path=str(TRAIN_PATH),
    test_path=str(TEST_PATH),
    model_name="multiclass_clinical",
    model_version="v0.4.0",
    use_pca=False,
    var_quantile=0.15,
    selector_on_log=False,
    pca_var_threshold=0.9,
    cv_splits=8,
    min_cancer_recall_for_threshold=0.9,
    threshold_objective="specificity",
    experiment_name="gse183635_multiclass_clinical",
    save_local_bundle=False,
    save_plots=False,
    # --- Features clínicas ---
    cluster_as_feature=True,
    malignant_cluster_col="mal_cluster",
    nm_cluster_col="nm_cluster",
    age_col="Age",
    sex_col="Sex",
)

# Feature grid (aplicamos aprendizajes de 4.0)
feat_grid = [
    dict(use_pca=False, selector_on_log=False, var_quantile=0.10, pca_var_threshold=0.9),
    dict(use_pca=False, selector_on_log=False, var_quantile=0.15, pca_var_threshold=0.9),
    dict(use_pca=False, selector_on_log=False, var_quantile=0.20, pca_var_threshold=0.9),
]

# Classifier grid
clf_grid = [
    ("logreg", dict(solver="lbfgs", max_iter=8500, C=0.5)),
    ("logreg", dict(solver="lbfgs", max_iter=8500, C=1.0)),
    ("logreg", dict(solver="lbfgs", max_iter=8500, C=2.0)),
    ("rf", dict(n_estimators=500, max_depth=None)),
    ("rf", dict(n_estimators=1000, max_depth=None)),
    ("extratrees", dict(n_estimators=1200, max_depth=None, min_samples_leaf=4)),
    ("extratrees", dict(n_estimators=1000, max_depth=None, min_samples_leaf=2)),
    ("extratrees", dict(n_estimators=800, max_depth=None)),
]

malignant_weights = [2.0, 3.0, 4.0, 6.0]

sweep = []
for feat_cfg in feat_grid:
    for clf_name, clf_params in clf_grid:
        for mw in malignant_weights:
            sweep.append(dict(
                feat_cfg=feat_cfg, clf_name=clf_name,
                clf_params=clf_params, mw=mw,
            ))

print(f"Total combinaciones: {len(sweep)}")

Total combinaciones: 96


: 

In [ ]:
results = []
errors = []

total = len(sweep)
start_all = perf_counter()

ema = None
alpha = 0.25
done = 0

pbar = tqdm(sweep, total=total, desc="Sweep Clinical", unit="run")

for combo in pbar:
    t0 = perf_counter()

    feat_cfg = combo["feat_cfg"]
    clf_name = combo["clf_name"]
    clf_params = combo["clf_params"]
    mw = combo["mw"]

    model_name = build_model_name(clf_name, feat_cfg, mw, variant_tag="clinical")
    cfg = replace(
        base_cfg,
        model_name=model_name,
        clf_name=clf_name,
        clf_params=clf_params,
        malignant_weight=mw,
        use_pca=feat_cfg["use_pca"],
        selector_on_log=feat_cfg["selector_on_log"],
        var_quantile=feat_cfg["var_quantile"],
        pca_var_threshold=feat_cfg["pca_var_threshold"],
    )

    try:
        out = run_training(cfg, feature_cols=gene_cols)
        tm = out["test_metrics"]
        results.append({
            "model_name": model_name,
            "clf_name": clf_name,
            "mw": mw,
            "use_pca": feat_cfg["use_pca"],
            "selector_on_log": feat_cfg["selector_on_log"],
            "var_quantile": feat_cfg["var_quantile"],
            "test_cancer_fn": tm["cancer_fn"],
            "test_cancer_fnr": tm["cancer_fnr"],
            "test_cancer_recall": tm["cancer_recall_sensitivity"],
            "test_cancer_specificity": tm["cancer_specificity"],
            "test_f1_macro": tm["f1_macro"],
            "test_balanced_accuracy": tm["balanced_accuracy"],
            "test_accuracy": tm["accuracy"],
            "mlflow_run_id": out["mlflow_run_id"],
        })
    except Exception as e:
        errors.append({
            "model_name": model_name,
            "clf_name": clf_name,
            "mw": mw,
            "use_pca": feat_cfg["use_pca"],
            "var_quantile": feat_cfg["var_quantile"],
            "error": repr(e),
        })

    dt = perf_counter() - t0
    ema = dt if ema is None else (alpha * dt + (1 - alpha) * ema)

    done += 1
    elapsed = perf_counter() - start_all
    remaining = (total - done) * (ema if ema is not None else 0.0)

    pbar.set_postfix({
        "last": fmt_secs(dt),
        "avg": fmt_secs(ema),
        "elapsed": fmt_secs(elapsed),
        "eta": fmt_secs(remaining),
        "ok": len(results),
        "err": len(errors),
    })

# DataFrames finales
res_df = (
    pd.DataFrame(results)
      .sort_values(
          ["test_cancer_fn", "test_cancer_fnr", "test_cancer_recall", "test_f1_macro"],
          ascending=[True, True, False, False],
      )
      .reset_index(drop=True)
)

err_df = pd.DataFrame(errors).reset_index(drop=True)

print(f"OK: {len(res_df)}, Errores: {len(err_df)}")

Sweep Clinical:   0%|          | 0/96 [00:00<?, ?run/s]2026/03/25 13:12:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/03/25 13:12:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/03/25 13:12:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/03/25 13:12:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/03/25 13:12:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/03/25 13:12:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/03/25 13:12:38 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/03/25 13:12:38 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/03/25 13:12:39 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/03/25 13:12:39 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/03/25 13:14:29 WARNING mlflow.models.model: `artifact_pat

In [ ]:
display(res_df)

In [ ]:
if len(err_df) > 0:
    display(err_df)

## Comparación con 4.0 (solo genes)

Resumen de las mejores métricas de cada experimento.

In [ ]:
print("=== Mejor modelo 4.2 (genes + clínicas + clusters como features) ===")
best_42 = res_df.iloc[0]
print(f"  Clasificador:      {best_42['clf_name']}")
print(f"  var_quantile:      {best_42['var_quantile']}")
print(f"  mw:                {best_42['mw']}")
print(f"  test_cancer_fn:    {best_42['test_cancer_fn']}")
print(f"  test_cancer_recall:{best_42['test_cancer_recall']:.4f}")
print(f"  test_f1_macro:     {best_42['test_f1_macro']:.4f}")
print(f"  test_bal_acc:      {best_42['test_balanced_accuracy']:.4f}")
print(f"  test_accuracy:     {best_42['test_accuracy']:.4f}")

print("\n--- Comparar con 4.0 ejecutando ambos notebooks ---")
print("Métricas de 4.0 se pueden consultar en MLflow o en el notebook 4.0.")

## Entrenamiento final (guardar bundle en `models/`)

In [ ]:
best = res_df.iloc[0].to_dict()
best

In [ ]:
best_cfg = replace(
    base_cfg,
    model_name="multiclass_clinical_final",
    model_version="v0.4.0",
    clf_name=best["clf_name"],
    malignant_weight=float(best["mw"]),
    use_pca=bool(best["use_pca"]),
    selector_on_log=bool(best["selector_on_log"]),
    var_quantile=float(best["var_quantile"]),
    save_local_bundle=True,
    save_plots=True,
    mlflow_log_artifacts=True,
    mlflow_log_model=True,
    output_figures_dir="reports/figures/multiclass_clinical",
)

final_result = run_training(best_cfg, feature_cols=gene_cols)

print("\nMÉTRICAS MODELO FINAL")
print("\n--- Test ---")
tm = final_result["test_metrics"]
for k, v in tm.items():
    if k == "per_class_report":
        continue
    print(f"  {k:30s}: {v}")